In [1]:
!pip install -U ultralytics
!python train_yolo26n_widerface.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]
python3: can't open file '/kaggle/working/train_yolo26n_widerface.py': [Errno 2] No such file or directory


In [2]:
"""
Train YOLO26n on WIDER FACE (single class: face) on Kaggle with 2x T4 GPUs (DDP) + image caching.

Quick start (Kaggle notebook: Accelerator = GPU T4 x2, Internet = On):
    !pip install -U ultralytics
    !python train_yolo26n_widerface.py

Common options:
    !python train_yolo26n_widerface.py --epochs 150 --imgsz 800 --batch 24
    !python train_yolo26n_widerface.py --time 10.5          # hard stop inside Kaggle's 12h limit
    !python train_yolo26n_widerface.py --resume             # continue an interrupted run
    !python train_yolo26n_widerface.py --cache ram          # force RAM cache (see note below)
    !python train_yolo26n_widerface.py --quiet              # one clean epoch bar instead of batch bars
    !python train_yolo26n_widerface.py --export onnx        # also export ONNX after training
    !python train_yolo26n_widerface.py --convert-only       # just build the dataset (+ cache) and exit

Add a WIDER FACE dataset to the notebook first (Add Input -> search "wider face"). It needs:
    WIDER_train/images/<event>/*.jpg        WIDER_val/images/<event>/*.jpg
    wider_face_split/wider_face_train_bbx_gt.txt
    wider_face_split/wider_face_val_bbx_gt.txt
The script finds these automatically under /kaggle/input.

Dual GPU:
    Runs DDP on GPU 0 and 1 by default (--device 0,1) and exits with a clear message if Kaggle
    gave you fewer than two GPUs. The batch size is split evenly (32 -> 16 per T4).

Caching (--cache auto | ram | disk | false, default auto):
    With DDP every GPU process builds its OWN cache, so a RAM cache costs (n_gpus x train set)
    + val set. At imgsz 640 that is ~12 GB x 2 + ~3 GB, which does not safely fit in Kaggle's
    ~29 GB. "auto" therefore measures your data and picks the best option that fits:
      1. ram  - if the per-GPU copies fit comfortably in RAM
      2. disk - decoded .npy files in /kaggle/working. All GPU processes read the SAME files, and
                the OS keeps them in the shared page cache, so you get RAM-speed reads with a
                single copy in memory (and no OOM risk: the OS can evict if it must)
      3. false - if neither fits (e.g. very large --imgsz)
    The dataset is also pre-resized once so its long side == imgsz (lossless for training, since
    Ultralytics resizes to imgsz anyway), which makes decoding and cache building much faster.

Progress you get:
    - tqdm bars for resizing/converting images and for building the disk cache
    - Ultralytics' per-batch bars during training (or, with --quiet, one tqdm epoch bar)
    - a one-line summary per epoch: loss, mAP50, mAP50-95, best so far, ETA
    - weights backed up to /kaggle/working/weights/ after EVERY epoch

Outputs:
    /kaggle/working/weights/yolo26n_widerface_best.pt
    /kaggle/working/weights/yolo26n_widerface_last.pt
    /kaggle/working/weights/metrics.json
    /kaggle/working/runs/yolo26n_widerface/      (logs, curves, confusion matrix)
"""

import argparse
import csv
import json
import os
import shutil
import subprocess
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from PIL import Image
from tqdm.auto import tqdm

# Kaggle has no InfiniBand; disabling it avoids slow NCCL start-up / hangs on multi-GPU runs.
os.environ.setdefault("NCCL_IB_DISABLE", "1")

# ----------------------------- constants -------------------------------------
INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
DATA_DIR = WORK / "datasets" / "widerface"   # converted YOLO-format dataset
RUNS_DIR = WORK / "runs"
WEIGHTS_DIR = WORK / "weights"
RUN_NAME = "yolo26n_widerface"
RUN_DIR = RUNS_DIR / RUN_NAME
DONE_MARKER = DATA_DIR / ".converted"
MIN_BOX = 2                    # skip boxes smaller than this (original pixels)
GB = 1024 ** 3
WORK_LIMIT_GB = 20             # Kaggle's /kaggle/working quota
PROC_OVERHEAD_GB = 3.5         # rough RAM per training process (torch, CUDA, workers)
# -----------------------------------------------------------------------------


def parse_args():
    p = argparse.ArgumentParser(description="Train YOLO26n on WIDER FACE (dual T4, cached)")
    p.add_argument("--model", default="yolo26n.pt", help="pretrained weights")
    p.add_argument("--epochs", type=int, default=100)
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--batch", type=int, default=32, help="total batch across all GPUs")
    p.add_argument("--workers", type=int, default=4)
    p.add_argument("--device", default="0,1", help="GPU ids, e.g. 0,1 | 0 | cpu")
    p.add_argument("--cache", default="auto", choices=["auto", "ram", "disk", "false"])
    p.add_argument("--time", type=float, default=None, help="max training hours (overrides epochs)")
    p.add_argument("--patience", type=int, default=30)
    p.add_argument("--resume", action="store_true", help="resume the last interrupted run")
    p.add_argument("--quiet", action="store_true", help="hide Ultralytics batch bars; show one epoch bar")
    p.add_argument("--export", default=None, help="export best weights, e.g. onnx, engine, openvino")
    p.add_argument("--convert-only", action="store_true", help="only build dataset (+disk cache), then exit")
    p.add_argument("--reconvert", action="store_true", help="rebuild the dataset from scratch")
    args, _ = p.parse_known_args()  # tolerant of extra args (e.g. when run inside a notebook)
    return args


# ================================= devices ====================================
def gpu_names():
    """GPU names via nvidia-smi, so the parent process never opens a CUDA context on GPU 0."""
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=15).stdout
        return [l.strip() for l in out.splitlines() if l.strip()]
    except Exception:
        return []


def resolve_device(arg):
    """Return (device_for_ultralytics, list_of_gpu_ids)."""
    if arg == "cpu":
        print("WARNING: training on CPU will be extremely slow.")
        return "cpu", []
    ids = [int(x) for x in str(arg).split(",")]
    names = gpu_names()
    if not names:
        sys.exit("ERROR: no CUDA GPU visible. In Kaggle set Accelerator = GPU T4 x2.")
    bad = [i for i in ids if i >= len(names)]
    if bad:
        sys.exit(f"ERROR: requested GPU(s) {bad} but only {len(names)} visible ({names}). "
                 f"In Kaggle set Accelerator = GPU T4 x2, or pass --device 0.")
    print(f"GPUs: {[f'{i}: {names[i]}' for i in ids]}")
    return (ids if len(ids) > 1 else ids[0]), ids


# ============================ dataset conversion ==============================
def find_annotation(name):
    hits = list(INPUT_ROOT.rglob(name))
    if not hits:
        sys.exit(f"ERROR: {name} not found under {INPUT_ROOT}. Did you add the WIDER FACE dataset?")
    return hits[0]


def parse_annotations(txt):
    """Return [(relative_image_path, [(x, y, w, h), ...]), ...] from a WIDER FACE bbx file."""
    lines = [l.strip() for l in txt.read_text().splitlines() if l.strip()]
    entries, i = [], 0
    while i < len(lines):
        rel, n = lines[i], int(lines[i + 1])
        k = max(n, 1)  # images with 0 faces still carry one dummy line
        boxes = []
        if n > 0:
            for l in lines[i + 2: i + 2 + k]:
                p = l.split()
                x, y, w, h = map(int, p[:4])
                if len(p) > 7 and int(p[7]) == 1:  # "invalid" flag
                    continue
                boxes.append((x, y, w, h))
        entries.append((rel, boxes))
        i += 2 + k
    return entries


def find_image_root(split_dir_name, sample_rel):
    for cand in INPUT_ROOT.rglob(split_dir_name):
        if (cand / "images" / sample_rel).exists():
            return cand / "images"
    sys.exit(f"ERROR: could not find images for {split_dir_name} under {INPUT_ROOT}")


def convert_split(split, ann_name, img_dir_name, imgsz):
    ann = find_annotation(ann_name)
    entries = parse_annotations(ann)
    img_root = find_image_root(img_dir_name, entries[0][0])
    tqdm.write(f"[{split}] {len(entries)} images | annotations: {ann} | images: {img_root}")

    img_out = DATA_DIR / "images" / split
    lbl_out = DATA_DIR / "labels" / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    def work(entry):
        rel, boxes = entry
        src = img_root / rel
        if not src.exists():
            return None
        name = Path(rel).name  # filenames are unique across events in WIDER FACE
        dst = img_out / name
        if dst.is_symlink():   # leftover from an older version of this script
            dst.unlink()
        with Image.open(src) as im:
            W, H = im.size     # original size: labels are normalised by this
            if not dst.exists():
                im.thumbnail((imgsz, imgsz), Image.Resampling.LANCZOS)  # only ever shrinks
                if im.mode != "RGB":
                    im = im.convert("RGB")
                tmp = dst.with_name(dst.name + ".tmp")
                im.save(tmp, format="JPEG", quality=95)
                tmp.replace(dst)
        lines, sides = [], []
        for x, y, w, h in boxes:
            x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
            bw, bh = x2 - x1, y2 - y1
            if bw < MIN_BOX or bh < MIN_BOX:
                continue
            lines.append(f"0 {(x1 + bw / 2) / W:.6f} {(y1 + bh / 2) / H:.6f} {bw / W:.6f} {bh / H:.6f}")
            sides.append(max(bw, bh))
        (lbl_out / f"{Path(name).stem}.txt").write_text("\n".join(lines))
        return sides

    with ThreadPoolExecutor(max_workers=min(16, (os.cpu_count() or 4) * 2)) as pool:
        results = list(tqdm(pool.map(work, entries), total=len(entries),
                            desc=f"Resizing+labels {split:<5}", unit="img", dynamic_ncols=True))

    missing = sum(r is None for r in results)
    sides = [s for r in results if r for s in r]
    n = max(len(sides), 1)
    tqdm.write(
        f"[{split}] {len(sides)} boxes | tiny(<16px): {100 * sum(s < 16 for s in sides) / n:.1f}% "
        f"| small(<32px): {100 * sum(s < 32 for s in sides) / n:.1f}% (original pixels)"
        + (f" | WARNING: {missing} images missing" if missing else "")
    )


def build_dataset(imgsz, reconvert=False):
    try:
        marker = json.loads(DONE_MARKER.read_text())
    except Exception:
        marker = None
    if DATA_DIR.exists() and (reconvert or (DONE_MARKER.exists() and (marker or {}).get("imgsz") != imgsz)):
        print("Rebuilding dataset (--reconvert or different --imgsz)...")
        shutil.rmtree(DATA_DIR)

    if DONE_MARKER.exists():
        print("Converted dataset found, skipping conversion.")
    else:
        convert_split("train", "wider_face_train_bbx_gt.txt", "WIDER_train", imgsz)
        convert_split("val", "wider_face_val_bbx_gt.txt", "WIDER_val", imgsz)
        for cache in (DATA_DIR / "labels").glob("*.cache"):  # stale Ultralytics label caches
            cache.unlink()
        DONE_MARKER.write_text(json.dumps({"imgsz": imgsz, "format": 2}))

    yaml_path = DATA_DIR / "widerface.yaml"
    yaml_path.write_text(
        f"path: {DATA_DIR}\ntrain: images/train\nval: images/val\nnames:\n  0: face\n"
    )
    return yaml_path


# ================================== caching ===================================
def _sample_bytes(split, imgsz, n=200):
    """Return (num_images, avg_ram_bytes, avg_disk_bytes) for a split, from a deterministic sample."""
    import random
    files = sorted((DATA_DIR / "images" / split).glob("*.jpg"))
    if not files:
        return 0, 0.0, 0.0
    sample = random.Random(0).sample(files, min(n, len(files)))
    ram = disk = 0.0
    for f in sample:
        with Image.open(f) as im:
            w, h = im.size
        r = imgsz / max(w, h)
        ram += int(w * r) * int(h * r) * 3   # Ultralytics RAM cache stores images resized to imgsz
        disk += w * h * 3                    # the .npy disk cache stores the decoded file as-is
    return len(files), ram / len(sample), disk / len(sample)


def _dir_size_excluding_npy(root):
    total = 0
    for dp, _, fns in os.walk(root):
        for fn in fns:
            if not fn.endswith(".npy"):
                try:
                    total += os.path.getsize(os.path.join(dp, fn))
                except OSError:
                    pass
    return total


def choose_cache(mode, imgsz, n_gpus):
    """Resolve --cache to 'ram' | 'disk' | False, printing the numbers behind the decision."""
    import psutil
    n_proc = max(n_gpus, 1)
    n_tr, tr_ram, tr_disk = _sample_bytes("train", imgsz)
    n_va, va_ram, va_disk = _sample_bytes("val", imgsz)
    # DDP: every process caches its own full train set; only rank 0 builds/caches the val set.
    ram_need = (n_tr * tr_ram) * n_proc + n_va * va_ram
    ram_overhead = PROC_OVERHEAD_GB * GB * n_proc + 1.5 * GB
    avail = psutil.virtual_memory().available
    disk_need = n_tr * tr_disk + n_va * va_disk
    used = _dir_size_excluding_npy(WORK) if WORK.exists() else 0
    disk_budget = min(WORK_LIMIT_GB * GB - used - 1.5 * GB,       # leave room for runs/weights
                      shutil.disk_usage(WORK if WORK.exists() else "/").free)

    print("\nCache estimate (imgsz={}):".format(imgsz))
    print(f"  RAM  : train {n_tr * tr_ram / GB:.1f} GB x {n_proc} process(es) + val {n_va * va_ram / GB:.1f} GB "
          f"= {ram_need / GB:.1f} GB (+{ram_overhead / GB:.1f} GB overhead) | available {avail / GB:.1f} GB")
    print(f"  Disk : {disk_need / GB:.1f} GB of .npy | budget {max(disk_budget, 0) / GB:.1f} GB "
          f"(shared page cache, single copy in memory)")

    ram_fits = ram_need + ram_overhead <= 0.92 * avail
    disk_fits = disk_need <= disk_budget

    if mode == "auto":
        choice = "ram" if ram_fits else "disk" if disk_fits else False
        print(f"  -> auto picked: {choice if choice else 'no cache'}"
              + ("" if ram_fits else "  (RAM copies per GPU would not fit safely)"))
        return choice
    if mode == "false":
        return False
    if mode == "ram" and not ram_fits:
        print("  WARNING: --cache ram probably exceeds available memory with one copy per GPU; "
              "risk of OOM-kill. Use --cache auto/disk or a smaller --imgsz.")
    if mode == "disk" and not disk_fits:
        print("  WARNING: --cache disk probably exceeds the disk budget; the run may fail with 'No space left'.")
    return mode


def prebuild_disk_cache():
    """Write decoded .npy files next to the images (what Ultralytics' cache='disk' expects)."""
    import cv2
    import numpy as np
    files = [f for s in ("train", "val") for f in sorted((DATA_DIR / "images" / s).glob("*.jpg"))]
    todo = [f for f in files if not f.with_suffix(".npy").exists()]
    if not todo:
        print("Disk cache already built.")
        return

    def work(f):
        tmp = f.with_suffix(".tmp.npy")
        np.save(tmp, cv2.imread(str(f)), allow_pickle=False)
        tmp.replace(f.with_suffix(".npy"))

    with ThreadPoolExecutor(max_workers=min(8, (os.cpu_count() or 4) * 2)) as pool:
        list(tqdm(pool.map(work, todo), total=len(todo), desc="Building disk cache",
                  unit="img", dynamic_ncols=True))


# ============================== epoch monitor =================================
class EpochMonitor(threading.Thread):
    """
    Watches results.csv and prints per-epoch progress. Also backs up weights every epoch.

    Runs in the main process, so it works even though multi-GPU (DDP) training runs in a
    subprocess where regular Ultralytics callbacks can't be attached.
    """

    def __init__(self, total_epochs, quiet, poll=5.0):
        super().__init__(daemon=True)
        self.total, self.quiet, self.poll = total_epochs, quiet, poll
        self.stop_event = threading.Event()
        rows = self.read_rows()
        self.seen = len(rows)  # non-zero when resuming
        self.best = max((self._metric(r, "metrics/mAP50-95") or 0.0 for r in rows), default=0.0)
        self.bar = None
        if quiet:
            self.bar = tqdm(total=total_epochs, initial=self.seen, desc="Training", unit="ep",
                            dynamic_ncols=True, leave=True)

    @staticmethod
    def _metric(row, prefix):
        for k, v in row.items():
            if k.startswith(prefix):
                try:
                    return float(v)
                except (TypeError, ValueError):
                    return None
        return None

    def read_rows(self):
        path = RUN_DIR / "results.csv"
        if not path.exists():
            return []
        try:
            with open(path, newline="") as f:
                rows = list(csv.DictReader(f))
        except OSError:
            return []
        return [{(k or "").strip(): (v or "").strip() for k, v in r.items()} for r in rows]

    def backup(self):
        WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
        for tag in ("best", "last"):
            src = RUN_DIR / "weights" / f"{tag}.pt"
            dst = WEIGHTS_DIR / f"{RUN_NAME}_{tag}.pt"
            try:
                if src.exists() and (not dst.exists() or src.stat().st_mtime > dst.stat().st_mtime):
                    shutil.copy2(src, dst)
            except OSError:
                pass  # file mid-write; we'll catch it next poll

    def report(self, rows):
        for i in range(self.seen, len(rows)):
            row = rows[i]
            ep = i + 1
            loss = 0.0
            for k, v in row.items():
                if k.startswith("train/"):
                    try:
                        loss += float(v)
                    except ValueError:
                        pass
            m50 = self._metric(row, "metrics/mAP50(")
            m5095 = self._metric(row, "metrics/mAP50-95")
            if m5095 is not None and m5095 > self.best:
                self.best = m5095
            try:
                elapsed = float(row.get("time", 0))
                eta = max(self.total - ep, 0) * elapsed / ep
                eta_s = f"ETA {int(eta // 3600)}h{int(eta % 3600 // 60):02d}m"
            except ValueError:
                eta_s = ""
            if self.bar is not None:
                self.bar.update(1)
                self.bar.set_postfix(loss=f"{loss:.3f}", mAP50=f"{m50 or 0:.3f}",
                                     mAP=f"{m5095 or 0:.3f}", best=f"{self.best:.3f}")
            else:
                tqdm.write(f"[Epoch {ep}/{self.total}] loss {loss:.3f} | mAP50 {m50 or 0:.4f} | "
                           f"mAP50-95 {m5095 or 0:.4f} | best {self.best:.4f} | {eta_s}")
        self.seen = max(self.seen, len(rows))

    def run(self):
        while not self.stop_event.wait(self.poll):
            self.report(self.read_rows())
            self.backup()

    def finish(self):
        self.stop_event.set()
        self.join(timeout=10)
        self.report(self.read_rows())
        self.backup()
        if self.bar is not None:
            self.bar.close()


# ================================== main ======================================
def main():
    args = parse_args()
    if args.quiet:
        os.environ["YOLO_VERBOSE"] = "False"  # must be set before importing ultralytics

    device, gpu_ids = resolve_device(args.device)
    n_gpus = len(gpu_ids)
    if n_gpus > 1 and args.batch % n_gpus:
        fixed = max(args.batch - args.batch % n_gpus, n_gpus)
        print(f"Batch {args.batch} is not divisible by {n_gpus} GPUs -> using {fixed}")
        args.batch = fixed

    data_yaml = build_dataset(args.imgsz, args.reconvert)
    cache = choose_cache(args.cache, args.imgsz, n_gpus)
    if cache == "disk":
        prebuild_disk_cache()
    if args.convert_only:
        print(f"Dataset ready: {data_yaml}")
        return

    import ultralytics
    from ultralytics import YOLO
    print(f"ultralytics {ultralytics.__version__} | device={device} | batch={args.batch} "
          f"({args.batch // max(n_gpus, 1)}/GPU) | imgsz={args.imgsz} | cache={cache}")

    if args.resume:
        last = RUN_DIR / "weights" / "last.pt"
        if not last.exists():
            sys.exit(f"ERROR: nothing to resume, {last} does not exist.")
        model = YOLO(str(last))
        train_kwargs = dict(resume=True)
    else:
        if RUN_DIR.exists():
            shutil.rmtree(RUN_DIR)  # stale logs would confuse the progress monitor
        model = YOLO(args.model)
        train_kwargs = dict(
            data=str(data_yaml), epochs=args.epochs, imgsz=args.imgsz, batch=args.batch,
            device=device, workers=args.workers, project=str(RUNS_DIR), name=RUN_NAME,
            exist_ok=True, amp=True, cache=cache, patience=args.patience, cos_lr=True,
            close_mosaic=10, fliplr=0.5, flipud=0.0, time=args.time, plots=True,
        )

    monitor = EpochMonitor(total_epochs=args.epochs, quiet=args.quiet)
    monitor.start()
    t0 = time.time()
    try:
        model.train(**train_kwargs)
    finally:
        monitor.finish()
    print(f"\nTraining finished in {(time.time() - t0) / 3600:.2f} h")

    # ---- final validation with the best weights (single GPU) ----
    best = WEIGHTS_DIR / f"{RUN_NAME}_best.pt"
    if not best.exists():
        sys.exit("ERROR: best weights were not saved - check the training log above.")
    print(f"Saved: {best}\nSaved: {WEIGHTS_DIR / f'{RUN_NAME}_last.pt'}")

    val_device = gpu_ids[0] if gpu_ids else "cpu"
    best_model = YOLO(str(best))
    m = best_model.val(data=str(data_yaml), imgsz=args.imgsz, batch=16, device=val_device)
    metrics = {
        "precision": float(m.box.mp), "recall": float(m.box.mr),
        "mAP50": float(m.box.map50), "mAP50-95": float(m.box.map),
        "imgsz": args.imgsz, "train_hours": round((time.time() - t0) / 3600, 3),
    }
    (WEIGHTS_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print("Final metrics:", json.dumps(metrics))

    # ---- optional export ----
    if args.export:
        out = Path(best_model.export(format=args.export, imgsz=args.imgsz))
        if out.exists():
            dst = WEIGHTS_DIR / out.name
            shutil.copy2(out, dst) if out.is_file() else shutil.copytree(out, dst, dirs_exist_ok=True)
            print(f"Exported: {dst}")


if __name__ == "__main__":
    main()

GPUs: ['0: Tesla T4', '1: Tesla T4']
[train] 12880 images | annotations: /kaggle/input/datasets/ichchhayakantnetweb/widerface/wider_face_split/wider_face_split/wider_face_train_bbx_gt.txt | images: /kaggle/input/datasets/ichchhayakantnetweb/widerface/WIDER_train/WIDER_train/images


Resizing+labels train:   0%|          | 0/12880 [00:00<?, ?img/s]

[train] 156985 boxes | tiny(<16px): 37.7% | small(<32px): 68.4% (original pixels)
[val] 3226 images | annotations: /kaggle/input/datasets/ichchhayakantnetweb/widerface/wider_face_split/wider_face_split/wider_face_val_bbx_gt.txt | images: /kaggle/input/datasets/ichchhayakantnetweb/widerface/WIDER_val/WIDER_val/images


Resizing+labels val  :   0%|          | 0/3226 [00:00<?, ?img/s]

[val] 39111 boxes | tiny(<16px): 38.5% | small(<32px): 67.3% (original pixels)

Cache estimate (imgsz=640):
  RAM  : train 10.4 GB x 2 process(es) + val 2.6 GB = 23.4 GB (+8.5 GB overhead) | available 29.9 GB
  Disk : 13.0 GB of .npy | budget 16.8 GB (shared page cache, single copy in memory)
  -> auto picked: disk  (RAM copies per GPU would not fit safely)


Building disk cache:   0%|          | 0/16106 [00:00<?, ?img/s]

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
ultralytics 8.4.197 | device=[0, 1] | batch=32 (16/GPU) | imgsz=640 | cache=disk
Ultralytics 8.4.197 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/datasets/widerface/widerface.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, dist

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      2/100      2.52G      2.269      1.333   0.003356        276        640: 0% ──────────── 0/403  0.3s[Epoch 1/100] loss 4.494 | mAP50 0.5270 | mAP50-95 0.2533 | best 0.2533 | ETA 5h57m
      2/100      5.99G      2.277      1.312   0.003912        391        640: 100% ━━━━━━━━━━━━ 403/403 3.3it/s 2:01
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 3.9it/s 13.0s
                   all       3226      39110      0.781      0.468      0.546      0.269

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/100      5.99G      2.272      1.371   0.003847        177        640: 0% ──────────── 3/403 3.0it/s 1.0s<2:12[Epoch 2/100] loss 3.593 | mAP50 0.5456 | mAP50-95 0.2685 | best 0.2685 | ETA 4h47m
      3/100         6G      2.274      1.311   0.003831        192        640: 100% ━━━━━━━━━━━━ 403/403 3.5it/s 1: